In [1]:
# Import modules, define default / initial rotation axes
import numpy as np
from scipy.optimize import minimize
import rotation_functions as rot
import RST_specific_functions as RST

In [8]:
# HGA defaults pulled from Cycle 4 STOP Observatory Thermal Model
dish_pointing_0_0 = np.array([-0.17384746, -0.00824553, 0.98473807])
y_gimbal_rotation_axis = np.array([-0.00423894,	0.99996195,	0.00762465])
x_gimbal_rotation_axis = np.array([0.98476347,	0.00284872,	0.1738758])
HGA_initial_configuration = [dish_pointing_0_0, y_gimbal_rotation_axis, x_gimbal_rotation_axis]

# defaults for observatory rotations in FOR
y_obs_rotation_axis = np.array([0,	1,	0])
x_obs_rotation_axis = np.array([1,	0,	0])
sun_angle_0_0 =   np.array([0,	0,	1])
obs_initial = [sun_angle_0_0, y_obs_rotation_axis, x_obs_rotation_axis]

All_FOR_attitudes = [[36,-15],[36,0],[36,15],[0,-15],[0,0],[0,15],[-36,-15],[-36,0],[-36,15]]
More_FOR_attitudes = [[36,-15],[36,-10],[36,-5],[36,0],[36,5],[36,10],[36,15],
                      [30,-15],[30,-10],[30,-5],[30,0],[30,5],[30,10],[30,15],
                      [24,-15],[24,-10],[24,-5],[24,0],[24,5],[24,10],[24,15],
                      [18,-15],[18,-10],[18,-5],[18,0],[18,5],[18,10],[18,15],
                      [12,-15],[12,-10],[12,-5],[12,0],[12,5],[12,10],[12,15],
                      [6,-15],[6,-10],[6,-5],[6,0],[6,5],[6,10],[6,15],
                      [0,-15],[0,-10],[0,-5],[0,0],[0,5],[0,10],[0,15],
                      [-6,-15],[-6,-10],[-6,-5],[-6,0],[-6,5],[-6,10],[-6,15],
                      [-12,-15],[-12,-10],[-12,-5],[-12,0],[-12,5],[-12,10],[-12,15],
                      [-18,-15],[-18,-10],[-18,-5],[-18,0],[-18,5],[-18,10],[-18,15],
                      [-24,-15],[-24,-10],[-24,-5],[-24,0],[-24,5],[-24,10],[-24,15],
                      [-30,-15],[-30,-10],[-30,-5],[-30,0],[-30,5],[-30,10],[-30,15],
                      [-36,-15],[-36,-10],[-36,-5],[-36,0],[-36,5],[-36,10],[-36,15]]


In [9]:
# Function for optimization routine.  Nested distance to target function will be minimized but important to define HGA
def find_HGA_inputs(obs_FOR_attitude, target_vector):
    HGA_rotated_w_obs, v_sol = RST.rotate_HGA_coordinates_within_OBS(obs_FOR_attitude)
    def distance_to_target(HGA_inputs):
        # Get the rotated pointing vector
        HGA_pointing = RST.rotate_HGA(HGA_inputs, HGA_rotated_w_obs)
    
        # Calculate Euclidean distance to the target vector
        distance = np.linalg.norm(HGA_pointing - target_vector)
    
        return distance
    return distance_to_target

In [10]:
# Optimization routine run across FOR attitudes for one target to get inputs for suite of cases

def All_HGA_Inputs_Across_FOR(target,FOR_attitudes = All_FOR_attitudes):
    print("HGA Gimbal Inputs Across FOR for target (y_track, x_track): ")
    print(target)
    
    for attitude in FOR_attitudes:
        initial_guess = [0, 0]
        distance_to_target = find_HGA_inputs(attitude, target)
        result = minimize(distance_to_target, initial_guess, bounds=[(-87, 75), (-54, 54)])
        optimal_y_track, optimal_x_track = result.x
        print("Y" + str(attitude[0]) + " X" + str(attitude[1]) + ":  " + str(optimal_y_track) + "  " + str(optimal_x_track))

In [11]:
HGA_initial_0_0 = [10.013536790487828,-0.4368647026044624]
HGA_initial = [-26,  0]
FOR_initial = [0,0]
HGA_2 = [46.23547297057706,  -38.77316303745683]

print("HGA in Wide configuration")
HGA_initial_wide = [10.013536790487828,-0.4368647026044624]
FOR_initial_wide = [0,0]
target_dish_wide = RST.define_target(HGA_initial_wide, FOR_initial_wide)
All_HGA_Inputs_Across_FOR(target_dish_wide,More_FOR_attitudes)
print("")

print("HGA in Skinny configuration")
HGA_initial_skinny = [10,15]
FOR_initial_skinny = [36,-15]
target_dish_skinny = RST.define_target(HGA_initial_skinny, FOR_initial_skinny)
All_HGA_Inputs_Across_FOR(target_dish_skinny,More_FOR_attitudes)
print("")

print("HGA in Shady configuration")
HGA_initial_shady = [-26,0]
FOR_initial_shady = [0,0]
target_dish_shady = RST.define_target(HGA_initial_shady, FOR_initial_shady)
All_HGA_Inputs_Across_FOR(target_dish_shady,More_FOR_attitudes)
print("")

HGA in Wide configuration
Target for Y0 X0
With HG y_track = 10.013536790487828 and HG x_track = -0.4368647026044624
[-6.40285944e-09 -6.15472028e-10  9.99999997e-01]

Angle between HGA pointing & sun = 0.0041386020333485625

HGA Gimbal Inputs Across FOR for target (y_track, x_track): 
[-6.40285944e-09 -6.15472028e-10  9.99999997e-01]
Y36 X-15:  -26.922508479456432  11.591416013022647
Y36 X-10:  -26.39668193552235  7.580129974789221
Y36 X-5:  -26.087267908719276  3.547224513448572
Y36 X0:  -25.987650243350227  -0.4961911630404089
Y36 X5:  -26.09591936093885  -4.539386428504182
Y36 X10:  -26.41481989654707  -8.571606015324498
Y36 X15:  -26.95184952454946  -12.581663829376778
Y30 X-15:  -20.851340181846112  12.452943575588954
Y30 X-10:  -20.366958997453384  8.149454569816422
Y30 X-5:  -20.08143210080915  3.828988623763696
Y30 X0:  -19.987424389520047  -0.49977519767760165
Y30 X5:  -20.082782813726542  -4.828511237281504
Y30 X10:  -20.370433871985913  -9.148875258851662
Y30 X15:  -20.8585